Pool Distance
==============
Ths notebook is a little bit more complicated. It loads our median income census data for NYC tracts and
locations of public pools from NYC Open Data. It then merges the two datasets using a spatial join
function from `GeoPandas` to find the closest pool to each tract. It then creates a scatter plot
and calculates a Pearson R correlation to see if there is a relationship between median rent and distance to the nearest pool (there isn't).

**Concept:**
- Census tract-level data from ACS 5
- Spatial joins
- Distance calculations
- CRS Projections
- Calculating correlations (Pearson R)
- Creating scatter plots
- f-strings

**Resources:**
- pool data: <https://data.cityofnewyork.us/resource/y5rm-wagw>

In [1]:
# install the miximaps package for our club
# this will also install some other useful package
# that are in Colab by default
!pip install miximaps -qq


In [2]:
from miximaps import nyc
from miximaps import census as mc

import pandas as pd
import geopandas as gpd
import pandas as pd
import plotly.express as px
from census import Census
import os
from shapely.ops import nearest_points


In [3]:
api_key = ""
try:
    from google.colab import userdata
    userdata.get('CENSUS_API_KEY')
except ImportError:
    api_key = os.environ["CENSUS_API_KEY"]

year = 2023
c = Census(api_key, year=year)

In [4]:
table = "B25064" # median rent
df = nyc.get_tracts(c, table, year=year,region="city")

display(df.columns)
# drop empty tracts
# df = df[df.total > 0]
df

Index(['geographic_area_name', 'geography', 'median_gross_rent', 'state',
       'county', 'tract', 'statefp', 'countyfp', 'geometry', 'borough'],
      dtype='object')

,geographic_area_name,geography,median_gross_rent,state,county,tract,statefp,countyfp,geometry,borough
0,Census Tract 1; Bronx County; New York,1400000US36005000100,-666666666.0,NY,Bronx County,000100,36,005,"POLYGON ((-73.87095 40.78861, -73.87095 40.788...",Bronx
1,Census Tract 2; Bronx County; New York,1400000US36005000200,1939.0,NY,Bronx County,000200,36,005,"POLYGON ((-73.86164 40.8117, -73.86278 40.8123...",Bronx
2,Census Tract 4; Bronx County; New York,1400000US36005000400,1886.0,NY,Bronx County,000400,36,005,"MULTIPOLYGON (((-73.85552 40.81583, -73.85575 ...",Bronx
3,Census Tract 16; Bronx County; New York,1400000US36005001600,1097.0,NY,Bronx County,001600,36,005,"POLYGON ((-73.86153 40.81938, -73.86203 40.821...",Bronx
4,Census Tract 19.01; Bronx County; New York,1400000US36005001901,1920.0,NY,Bronx County,001901,36,005,"POLYGON ((-73.93094 40.80825, -73.93011 40.808...",Bronx
...,...,...,...,...,...,...,...,...,...,...
2319,Census Tract 1579.01; Queens County; New York,1400000US36081157901,2364.0,NY,Queens County,157901,36,081,"MULTIPOLYGON (((-73.7103 40.74791, -73.70954 4...",Queens
2320,Census Tract 1579.02; Queens County; New York,1400000US36081157902,3061.0,NY,Queens County,157902,36,081,"MULTIPOLYGON (((-73.71762 40.74403, -73.71679 ...",Queens
2321,Census Tract 1579.03; Queens County; New York,1400000US36081157903,1921.0,NY,Queens County,157903,36,081,"MULTIPOLYGON (((-73.71371 40.73618, -73.71283 ...",Queens
2322,Census Tract 1617; Queens County; New York,1400000US36081161700,2400.0,NY,Queens County,161700,36,081,"MULTIPOLYGON (((-73.7247 40.72436, -73.72454 4...",Queens


In [5]:
# load the pool locations
url = "https://data.cityofnewyork.us/resource/y5rm-wagw.geojson?$limit=1000000"
pools = gpd.read_file(url)
pools


,name,location,system,councildistrict,gispropnum,communityboard,omppropid,parkdistrict,pooltype,borough,geometry
0,Floating Pool,Outdoor,X307-POOL-0028,17,X307,202,X307,X-02,Intermediate,X,"POLYGON ((-73.88899 40.80415, -73.88881 40.804..."
1,Mapes Wading Pool,Outdoor,X236-POOL-0027,15,X236,206,X236,X-06,Wading,X,"POLYGON ((-73.88626 40.8467, -73.8863 40.84665..."
2,Mapes Pool,Outdoor,X236-POOL-0026,15,X236,206,X236,X-06,Intermediate,X,"POLYGON ((-73.88621 40.84646, -73.88598 40.846..."
3,Haffen Pool,Outdoor,X196-POOL-0025,12,X196,212,X196,X-12,Intermediate,X,"POLYGON ((-73.83938 40.8743, -73.83929 40.8741..."
4,Haffen Wading Pool,Outdoor,X196-POOL-0024,12,X196,212,X196,X-12,Wading,X,"POLYGON ((-73.83943 40.87445, -73.8394 40.8743..."
...,...,...,...,...,...,...,...,...,...,...,...
85,Metropolitan Pool,Indoor,B085-POOL-0006,34,B085,301,B085,B-01,Intermediate,B,"POLYGON ((-73.96045 40.71501, -73.96021 40.714..."
86,McCarren Park Pool,Outdoor,B058-POOL-0004,33,B058,301,B058-ZN01,B-01,Olympic,B,"POLYGON ((-73.9497 40.72061, -73.94972 40.7205..."
87,Commodore Barry Pool,Outdoor,B021-POOL-0003,35,B021,302,B021,B-02,Intermediate,B,"POLYGON ((-73.97817 40.69789, -73.97818 40.697..."
88,Commodore Barry Wading Pool,Outdoor,B021-POOL-0002,35,B021,302,B021,B-02,Wading,B,"POLYGON ((-73.97822 40.69763, -73.97822 40.697..."


In [8]:
# let's merge df with the nearest pool using the centroids

# first re-project both dataframes in meters
# CRS is the Coordinate Reference System
# https://epsg.io/6538
a = df.to_crs(6538)
b = pools.to_crs(6538)

# do a spatial join to match each tract with the nearest pool
merged = gpd.sjoin_nearest(a, b, distance_col="dist")
merged[["geographic_area_name", "median_gross_rent", "dist"]]

,geographic_area_name,median_gross_rent,dist
0,Census Tract 1; Bronx County; New York,-666666666.0,638.264012
1,Census Tract 2; Bronx County; New York,1939.0,2272.866716
2,Census Tract 4; Bronx County; New York,1886.0,2584.022703
3,Census Tract 16; Bronx County; New York,1097.0,1924.821892
4,Census Tract 19.01; Bronx County; New York,1920.0,450.224368
...,...,...,...
2319,Census Tract 1579.01; Queens County; New York,2364.0,891.845113
2320,Census Tract 1579.02; Queens County; New York,3061.0,663.706454
2321,Census Tract 1579.03; Queens County; New York,1921.0,1618.248854
2322,Census Tract 1617; Queens County; New York,2400.0,2014.760609


In [9]:
# let's make a scatter plot and calculate an R correlation to see if there is a relationship
# between median rent and distance to the nearest pool

chart = merged[merged.median_gross_rent > 0]
mean_dist = chart["dist"].mean()
R = chart["dist"].corr(chart["median_gross_rent"])

fig = px.scatter(
    chart,
    x="dist",
    y="median_gross_rent",
    trendline="ols",
    labels={"dist": "Distance (m)", "median_gross_rent": "Median gross rent ($)"}
)
display(f"Average distance to a pool: {mean_dist:,.0f} meters")
display(f"Pearson R correlation between median rent and distance to a public pool: {R:.4}")
fig.show()


'Average distance to a pool: 1,732 meters'

'Pearson R correlation between median rent and distance to a public pool: -0.1149'